# Marginal distributions of the MI pruning variables

[mi_importance.py](mi_importance.py) scores channels with a closed-form Gaussian conditional MI.
That formula is exact only if the variables it consumes are **jointly Gaussian**. Gaussian
*marginals* are a necessary condition for that, so this notebook measures the marginals first,
across the whole network.

Three kinds of variable go into the estimator:

| variable | one sample is | one column is |
|---|---|---|
| **per-pixel activation** | an (image, pixel) pair | a conv channel's value at that pixel |
| **pooled activation** | an image | a conv channel averaged over its whole map |
| **model output** | an image | one cell of the predicted noise, pooled to 3×8×8 |

The first two exist at every conv; the third only at the output. The plan:

1. capture activations,
2. measure skew and excess kurtosis of **every channel of every layer**,
3. look at the histogram shapes behind a range of those values,
4. see how skew and kurtosis are distributed, and how they change with depth.

Multivariate tests come after — marginals passing does not establish joint Gaussianity.

## 1. Setup

In [ ]:
!git clone --branch mipp-lookahead2 https://github.com/elliotcanter11/Diff-Pruning.git

In [ ]:
%cd Diff-Pruning/

In [ ]:
!pip install -r requirements.txt

In [ ]:
!python tools/extract_cifar10_hug.py --output data

In [ ]:
!bash tools/convert_cifar10_ddpm_ema.sh

## 2. Capture activations

The same calibration loop `ddpm_prune.py` runs before pruning: draw images, draw a random timestep
per image, add noise, forward the pretrained model, and record every conv's output. So the numbers
below are exactly what the estimator sees.

In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.auto import tqdm

# compat shims, copied from ddpm_prune.py (diffusers/ here is the vendored copy)
import huggingface_hub
from huggingface_hub import constants as hf_constants
if not hasattr(hf_constants, "hf_cache_home"):
    hf_constants.hf_cache_home = hf_constants.HF_HUB_CACHE
if not hasattr(huggingface_hub, "cached_download"):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download
if not hasattr(huggingface_hub, "HfFolder"):
    class HfFolder:
        @staticmethod
        def get_token(): return huggingface_hub.get_token()
    huggingface_hub.HfFolder = HfFolder
import jax
if not hasattr(jax.random, "KeyArray"): jax.random.KeyArray = jax.Array
import transformers.utils as tf_utils
if not hasattr(tf_utils, "FLAX_WEIGHTS_NAME"): tf_utils.FLAX_WEIGHTS_NAME = "flax_model.msgpack"

from diffusers import DDPMPipeline
from mi_importance import MIImportance
from torchvision import transforms as T
import utils

DEVICE  = 'cuda:0'
BATCH   = 128
BATCHES = 16     # BATCH * BATCHES images
LOCS    = 4      # pixels sampled per image  (= --mi_num_locations)

pipeline = DDPMPipeline.from_pretrained('pretrained/ddpm_ema_cifar10').to(DEVICE)
model, scheduler = pipeline.unet.eval(), pipeline.scheduler

tf = T.Compose([T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean=0.5, std=0.5)])
loader = torch.utils.data.DataLoader(
    utils.get_dataset('data/cifar10_images', transform=tf),
    batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)

imp = MIImportance(num_locations=LOCS).attach(model, ignored_layers=[model.conv_out])

it = iter(loader)
with torch.no_grad():
    for _ in tqdm(range(BATCHES), desc='capturing'):
        batch = next(it)
        batch = batch[0] if isinstance(batch, (list, tuple)) else batch
        batch = batch.to(DEVICE)
        t = torch.randint(0, scheduler.config.num_train_timesteps,
                          (batch.shape[0],), device=DEVICE).long()
        noisy = scheduler.add_noise(batch, torch.randn_like(batch), t)
        imp.new_pass(batch.shape[0])
        out = model(noisy, t).sample
        imp.record_output(out)
        imp.record_timesteps(t)
imp.finalize()

convs = [m for m, _ in sorted(imp._order.items(), key=lambda kv: kv[1])
         if imp._loc_buf.get(m) is not None]
name_of = {m: n for n, m in model.named_modules()}

loc = lambda m: imp._loc_buf[m].float().numpy()   # (n_images*LOCS, C)
img = lambda m: imp._img_buf[m].float().numpy()   # (n_images, C)
out_target = imp._out_target.numpy()              # (n_images, 3*8*8)

n_img = out_target.shape[0]
print(f'{len(convs)} convs captured')
print(f'{n_img} images  ->  {n_img*LOCS} per-pixel rows, {n_img} per-image rows')

## 3. Measure skew and kurtosis everywhere

For each of the three variable types, take the **skew** and **excess kurtosis** of every column.
A Gaussian has 0 for both: skew measures asymmetry, excess kurtosis measures how heavy the tails are
relative to a Gaussian.

Nothing here is restricted to one layer: `per-pixel activation` and `pooled activation` cover every
channel of every conv.

In [ ]:
# Three variable types. For the two per-layer ones, the list is indexed by conv execution order.
SAMPLES = {
    'per-pixel activation': [loc(m) for m in convs],
    'pooled activation':    [img(m) for m in convs],
    'model output':         [out_target],
}
COLORS = {'per-pixel activation': '#3b6fb6',
          'pooled activation':    '#e8a33d',
          'model output':         '#4c9f70'}
RED, GREY = '#d1495b', '#8d99ae'

plt.rcParams.update({'figure.dpi': 120, 'font.size': 8.5,
                     'axes.spines.top': False, 'axes.spines.right': False})

def moments(A):
    """Skew and excess kurtosis of each column. Constant columns have no shape, so drop them."""
    A = A.astype(np.float64)
    keep = A.std(0) > 0
    s, k = stats.skew(A[:, keep], 0), stats.kurtosis(A[:, keep], 0)
    good = np.isfinite(s) & np.isfinite(k)
    return np.flatnonzero(keep)[good], s[good], k[good]

LAYER, CHAN, SKEW, KURT = 0, 1, 2, 3          # column indices of the tables below

MOMENTS, dropped = {}, 0
for name, blocks in SAMPLES.items():
    rows = []
    for li, A in enumerate(blocks):
        ch, s, k = moments(A)
        dropped += A.shape[1] - len(ch)
        rows.append(np.column_stack([np.full(len(ch), li), ch, s, k]))
    MOMENTS[name] = np.vstack(rows)           # one row per column: layer, channel, skew, kurt

def values(name, li, ci):
    """The raw samples behind one (layer, channel) entry of a MOMENTS table."""
    return SAMPLES[name][int(li)][:, int(ci)].astype(np.float64)

for name, M in MOMENTS.items():
    print(f'{name:<22} {len(M):>6} columns over {len(SAMPLES[name]):>2} layer(s)   '
          f'median kurt {np.median(M[:, KURT]):+7.2f}   median skew {np.median(M[:, SKEW]):+.2f}')
print(f'\n({dropped} constant columns dropped)')

## 4. What those numbers look like

Each panel is **one channel**, standardised to mean 0 and variance 1, so only the *shape* is being
compared. Red is the standard normal. The channels are chosen to span the range of each statistic,
picked from all layers at once.

All panels share the same axes, so a bar clipping at the top means a peak taller than the plot —
the printed value says by how much.

In [ ]:
grid = np.linspace(-4, 4, 400)

def shape_grid(stat, label, quantiles=(0.02, 0.25, 0.50, 0.75, 0.98)):
    """One row per variable type; columns step through the quantiles of `stat`."""
    fig, axes = plt.subplots(len(SAMPLES), len(quantiles),
                             figsize=(2.05*len(quantiles), 1.75*len(SAMPLES)), sharex=True)
    for r, name in enumerate(SAMPLES):
        M = MOMENTS[name]
        order = np.argsort(M[:, stat])
        for c, q in enumerate(quantiles):
            row = M[order[int(round(q*(len(order) - 1)))]]
            x = values(name, row[LAYER], row[CHAN])
            x = (x - x.mean()) / x.std()
            counts, edges = np.histogram(x, bins=60, range=(-4, 4))
            ax = axes[r, c]
            ax.stairs(counts / (len(x)*(edges[1] - edges[0])), edges, fill=True,
                      color=COLORS[name], alpha=.65)
            ax.plot(grid, stats.norm.pdf(grid), color=RED, lw=1.2)
            ax.set_title(f'{label} {row[stat]:+.2f}', fontsize=8.5, pad=3)
            tag = (f'layer {int(row[LAYER])}  ch {int(row[CHAN])}' if len(SAMPLES[name]) > 1
                   else f'cell {int(row[CHAN])}')          # model output has only one "layer"
            ax.text(.04, .94, tag, color=GREY, transform=ax.transAxes, fontsize=6.5, va='top')
            ax.set_xlim(-4, 4); ax.set_xticks([-2, 0, 2])
            ax.set_ylim(0, 2*stats.norm.pdf(0)); ax.set_yticks([])
        axes[r, 0].set_ylabel(name.replace(' ', '\n'), fontsize=7.5)
    fig.supxlabel('standardised value', fontsize=8.5)
    return fig

fig = shape_grid(KURT, 'kurt')
fig.suptitle('Shapes from lightest to heaviest tails   (red = standard normal)', y=1.01, fontsize=10)
fig.tight_layout(); plt.show()

In [ ]:
fig = shape_grid(SKEW, 'skew')
fig.suptitle('Shapes from most left-skewed to most right-skewed   (red = standard normal)',
             y=1.01, fontsize=10)
fig.tight_layout(); plt.show()

## 5. How those numbers are distributed

**Left** — what fraction of all channels fall below a given value. Read the height where a curve
crosses a value: if the blue curve passes 0.5 at kurtosis 8, half of all per-pixel channels are
heavier-tailed than that.

**Right** — the same statistic against depth: median over each layer's channels, with the
interquartile range shaded. This is what says whether any single layer is representative.
`model output` has no depth, so it appears as a dashed level line.

The red line marks the Gaussian value, 0. Kurtosis uses a **symlog** scale (linear within ±1,
logarithmic outside) because it spans orders of magnitude; skew is small enough to plot linearly.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 5.4))
for r, (stat, label) in enumerate([(SKEW, 'skew'), (KURT, 'excess kurtosis')]):
    ecdf, depth = axes[r, 0], axes[r, 1]

    for name, M in MOMENTS.items():
        v = np.sort(M[:, stat])
        ecdf.plot(v, np.arange(1, len(v) + 1)/len(v), color=COLORS[name], lw=1.6, label=name)

        if len(SAMPLES[name]) == 1:            # model output has no depth -- show as a level
            depth.axhline(np.median(M[:, stat]), color=COLORS[name], lw=1.4, ls='--')
            continue
        li = M[:, LAYER]
        xs = np.array([i for i in range(len(SAMPLES[name])) if (li == i).any()])
        qs = np.array([np.percentile(M[li == i, stat], [25, 50, 75]) for i in xs])
        depth.fill_between(xs, qs[:, 0], qs[:, 2], color=COLORS[name], alpha=.22, lw=0)
        depth.plot(xs, qs[:, 1], color=COLORS[name], lw=1.6)

    # skew sits in single digits, kurtosis spans orders of magnitude -- scale each to suit
    if stat is KURT:
        ecdf.set_xscale('symlog', linthresh=1)
        depth.set_yscale('symlog', linthresh=1)
        note = ' (symlog)'
    else:
        ecdf.set_xlim(-4, 4)                  # a few channels lie outside; the curve leaves
        note = ''                             # the frame short of 0 or 1 where they do
    ecdf.axvline(0, color=RED, lw=1.1); depth.axhline(0, color=RED, lw=1.1)
    ecdf.set_xlabel(f'{label}   (0 = Gaussian){note}'); ecdf.set_ylabel('fraction of channels')
    depth.set_xlabel('conv index, execution order'); depth.set_ylabel(label + note)

axes[0, 0].legend(fontsize=7.5, frameon=False, loc='upper left')
fig.suptitle('Skew and kurtosis over every channel of every layer', y=1.0, fontsize=10)
fig.tight_layout(); plt.show()

## What this does and does not settle

A variable type whose curves sit on 0 has Gaussian-looking marginals. That is **necessary but not
sufficient** — a set of variables can have perfectly Gaussian marginals and still be far from jointly
Gaussian, and the joint is what the determinant formula actually uses.

Two things to keep in mind when reading the figures above:

* Every histogram pools all timesteps. A mixture of Gaussians with different scales is heavy-tailed
  even when each individual timestep is perfectly Gaussian, so heavy tails here do not yet say which
  is happening.
* `pooled activation` and `model output` are averages over H×W, so the central limit theorem pushes
  them toward Gaussian regardless. They get an easier test than `per-pixel activation` does.

Multivariate tests are next.